In [0]:
from datetime import datetime
import pandas as pd

checks = [
    ("event_id_unique_in_silver",
     """SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM workspace.silver.order_events""",
     "duplicates removed"),
    ("bronze_to_silver_event_reconciliation",
     """SELECT (SELECT COUNT(DISTINCT event_id) FROM workspace.bronze.order_events) - (SELECT COUNT(*) FROM workspace.silver.order_events)""",
     "no real event lost while de-duplicating"),
    ("orders_row_count_matches",
     """SELECT (SELECT COUNT(*) FROM workspace.bronze.orders) - (SELECT COUNT(*) FROM workspace.silver.orders)""",
     "no order lost or fanned out by joins"),
    ("order_items_row_count_matches",
     """SELECT (SELECT COUNT(*) FROM workspace.bronze.order_items) - (SELECT COUNT(*) FROM workspace.silver.order_items)""",
     "no line lost or duplicated by the join to events"),
    ("no_incomplete_orders",
     """SELECT COUNT(*) FROM workspace.silver.orders WHERE order_status = 'incomplete'""",
     "every order is either delivered or cancelled"),
    ("shipment_stage_order_is_valid",
     """SELECT COUNT(*) FROM workspace.silver.sub_order_timeline WHERE delivered_ts IS NOT NULL AND NOT (picking_started_ts <= packing_completed_ts AND packing_completed_ts <= picked_up_ts AND picked_up_ts <= delivered_ts)""",
     "timestamps follow the physical process"),
    ("delivered_not_before_placed",
     """SELECT COUNT(*) FROM workspace.silver.orders WHERE delivered_ts < placed_ts""",
     "no time travel"),
    ("orphan_events",
     """SELECT COUNT(*) FROM workspace.silver.order_events e LEFT JOIN workspace.bronze.orders o ON e.order_id = o.order_id WHERE o.order_id IS NULL""",
     "every event belongs to an order"),
    ("fulfilled_not_above_ordered",
     """SELECT COUNT(*) FROM workspace.silver.order_items WHERE qty_fulfilled > qty_ordered""",
     "quantities are sane"),
]

rows = []
for name, query, why in checks:
    violations = int(spark.sql(query).collect()[0][0])
    rows.append({"check_name": name, "violations": violations,
                 "status": "PASS" if violations == 0 else "FAIL",
                 "why_it_matters": why, "run_ts": datetime.now()})

dq = pd.DataFrame(rows)
display(dq)
spark.createDataFrame(dq).write.mode("append").saveAsTable("workspace.silver.dq_results")
assert (dq["status"] == "PASS").all(), "Data quality checks failed, see the table above"

In [0]:
%sql
SELECT COUNT(*) AS rider_events_still_missing_rider
FROM workspace.silver.order_events
WHERE event_type IN ('rider_assigned','rider_arrived_at_store','order_picked_up','order_delivered')
  AND rider_id IS NULL;